In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import glob
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# 1. ĐỌC TRỰC TIẾP DỮ LIỆU TỪ READ-SPLITS-V1
train_files = glob.glob("/kaggle/input/**/train_baseline_all.parquet", recursive=True)
test_files = glob.glob("/kaggle/input/**/test_9600.parquet", recursive=True)
assert len(train_files) > 0 and len(test_files) > 0, "Chưa gắn dataset read-splits-v1 vào Input!"

df_train = pd.read_parquet(train_files[0]).dropna(subset=["text", "label"]).copy()
df_test = pd.read_parquet(test_files[0]).dropna(subset=["text", "label"]).copy()

df_train["text"] = df_train["text"].astype(str).str.strip()
df_test["text"] = df_test["text"].astype(str).str.strip()
df_train = df_train[df_train["text"].str.len() > 0].reset_index(drop=True)
df_test = df_test[df_test["text"].str.len() > 0].reset_index(drop=True)

label2id = {"Human": 0, "AI": 1}
df_train["labels"] = df_train["label"].map(label2id).astype(int)
df_test["labels"] = df_test["label"].map(label2id).astype(int)

# Chia phân tầng an toàn
stratify_col = df_train["domain"].fillna("unknown").astype(str) + "_" + df_train["labels"].astype(str)
rare = stratify_col.value_counts()
stratify_target = df_train["labels"] if (rare < 2).any() else stratify_col

df_tr, df_val = train_test_split(df_train, test_size=0.1, random_state=42, stratify=stratify_target)
df_tr = df_tr.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
print(f"Train={len(df_tr):,} | Val={len(df_val):,} | Test={len(df_test):,}", flush=True)

# 2. TOKENIZE NHANH VỚI ROBERTA-LARGE (~20 GIÂY)
MODEL_ID = "roberta-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

ds_tr = Dataset.from_pandas(df_tr[["text", "labels"]]).map(tokenize_fn, batched=True, remove_columns=["text"])
ds_val = Dataset.from_pandas(df_val[["text", "labels"]]).map(tokenize_fn, batched=True, remove_columns=["text"])
ds_test = Dataset.from_pandas(df_test[["text", "labels"]]).map(tokenize_fn, batched=True, remove_columns=["text"])

# 3. HÀM ĐÁNH GIÁ CHỈ SỐ CHUẨN
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "auroc": roc_auc_score(labels, probs),
        "tpr_ai": tp / max(tp + fn, 1),
        "fpr_human_as_ai": fp / max(fp + tn, 1),
    }

# 4. HUẤN LUYỆN BASELINE ROBERTA-LARGE (FP16 ỔN ĐỊNH TUYỆT ĐỐI)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

out_dir = "/kaggle/working/baseline_roberta"
os.makedirs(out_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=f"{out_dir}/checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1.5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=250,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=True,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    logging_steps=200,
    disable_tqdm=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tr,
    eval_dataset=ds_val,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

# 5. ĐÁNH GIÁ TRÊN TẬP TEST 9,600 MẪU & LƯU KẾT QUẢ
test_out = trainer.predict(ds_test)
probs_ai = torch.softmax(torch.tensor(test_out.predictions, dtype=torch.float32), dim=-1)[:, 1].numpy()
preds_int = (probs_ai >= 0.5).astype(int)

keep_cols = [c for c in ["sample_id", "domain", "sub_source", "source_dataset", "label"] if c in df_test.columns]
df_test_res = df_test[keep_cols].copy()
df_test_res["prob_ai"] = probs_ai
df_test_res["pred_label"] = np.where(preds_int == 1, "AI", "Human")
df_test_res.to_parquet(f"{out_dir}/roberta_test_predictions.parquet", index=False)

print("\n=== KẾT QUẢ BASELINE ROBERTA-LARGE TRÊN TEST 9,600 ===", flush=True)
for k, v in test_out.metrics.items():
    if k.startswith("test_"):
        print(f"{k.replace('test_', ''):18s}: {v:.4f}", flush=True)

Train=39,600 | Val=4,400 | Test=9,600


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (8 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.3963', 'grad_norm': '76.5', 'learning_rate': '7.505e-06', 'epoch': '1.051'}
{'loss': '0.3432', 'grad_norm': '0.6354', 'learning_rate': '6.505e-06', 'epoch': '1.131'}
{'loss': '0.3253', 'grad_norm': '1.318', 'learning_rate': '5.523e-06', 'epoch': '1.212'}
{'loss': '0.2838', 'grad_norm': '14.41', 'learning_rate': '4.577e-06', 'epoch': '1.293'}
{'loss': '0.3141', 'grad_norm': '14.62', 'learning_rate': '3.682e-06', 'epoch': '1.374'}
{'loss': '0.3132', 'grad_norm': '179.7', 'learning_rate': '2.856e-06', 'epoch': '1.455'}
{'loss': '0.3282', 'grad_norm': '0.6444', 'learning_rate': '2.112e-06', 'epoch': '1.535'}
{'loss': '0.2627', 'grad_norm': '79.42', 'learning_rate': '1.465e-06', 'epoch': '1.616'}
{'loss': '0.334', 'grad_norm': '127.9', 'learning_rate': '9.253e-07', 'epoch': '1.697'}
{'loss': '0.2233', 'grad_norm': '1.823', 'learning_rate': '5.029e-07', 'epoch': '1.778'}
{'loss': '0.2575', 'grad_norm': '142', 'learning_rate': '2.055e-07', 'epoch': '1.859'}
{'loss': '0.2808', 'gra

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '5335', 'train_samples_per_second': '14.85', 'train_steps_per_second': '0.928', 'train_loss': '0.4704', 'epoch': '2'}

=== KẾT QUẢ BASELINE ROBERTA-LARGE TRÊN TEST 9,600 ===
loss              : 0.3137
accuracy          : 0.8848
macro_f1          : 0.8844
auroc             : 0.9678
tpr_ai            : 0.9450
fpr_human_as_ai   : 0.1755
runtime           : 182.5390
samples_per_second: 52.5910
steps_per_second  : 1.6430
